# Train the integer GPT-2 on Colab

Runs `scripts/train.py` on a Colab GPU (Runtime → Change runtime type → GPU) and downloads the checkpoints. Training only: compiling and hovering happen locally.

1. Run the upload cell and pick `scripts/train.py` from your checkout.
2. Edit `RUNS` if you want other shapes.
3. Put the downloaded JSON files in `model/exp/`, then locally: `QAT=model/exp/<name>.json node scripts/compile.ts` and `node scripts/generate.ts "Once upon a time" 48`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0))
from google.colab import files
import os
os.makedirs('scripts', exist_ok=True); os.makedirs('model/exp', exist_ok=True)
up = files.upload()  # choose scripts/train.py
open('scripts/train.py', 'wb').write(up['train.py'])

In [ ]:
# name -> environment overrides (see the docstring of scripts/train.py)
RUNS = {
    'ts-d64-l4-b128': dict(BLOCK=128),
    'ts-d64-l4-b256': dict(BLOCK=256),
}

In [ ]:
import subprocess, sys, os
for name, env in RUNS.items():
    out = f'model/exp/{name}.json'
    print(f'=== {name} {env}', flush=True)
    e = {**os.environ, **{k: str(v) for k, v in env.items()}, 'QAT_OUT': out}
    with open(f'model/exp/{name}.log', 'w') as log:
        p = subprocess.Popen([sys.executable, 'scripts/train.py'], env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end=''); log.write(line)
        assert p.wait() == 0, f'{name} failed'

In [ ]:
import shutil
shutil.make_archive('checkpoints', 'zip', 'model/exp')
files.download('checkpoints.zip')